# View CosMx Landscape (DegaFiles hosted on the workbench bucket)
Load the CosMx DegaFiles produced by `CosMx_pre-process.ipynb` into the Celldega `Landscape` widget, reading them directly from the workbench S3 bucket (so it works on Manifold without a local server).

The workbench is an s3fs mount of `s3://$S3_BUCKET/research/projects/$PROJECT_ID/` — so files under `workbench/CosMx_Celldega/cosmx_dega_files/S0` are served at the S3 https URL built below. Pass temporary AWS creds via the `creds` traitlet.

Defaults to `SAMPLE = 'S0_non-row-group'` (classic one-file-per-tile layout), which streams from S3 with temporary creds. The row-group `'S0'` build issues chunked-parquet HTTP Range requests that currently return 403 against this bucket, so prefer the non-row-group build for bucket hosting.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import boto3
import celldega as dega
print(dega.__version__)

In [ ]:
# Build the S3 https base_url for the DegaFiles on the workbench bucket.
# Use the non-row-group build: its classic single-file-per-tile layout streams from S3 with
# temporary creds (the row-group 'S0' build issues chunked-parquet Range requests that 403).
SAMPLE = 'S0_non-row-group'  # or 'S0' (row-group; see note above)
base_url = (
    'https://' + os.environ['S3_BUCKET'] + '.s3.us-east-1.amazonaws.com/'
    + 'research/projects/' + os.environ['PROJECT_ID']
    + '/CosMx_Celldega/cosmx_dega_files/' + SAMPLE
)
creds = boto3.Session().get_credentials().get_frozen_credentials()
print(base_url)

In [ ]:
landscape = dega.viz.Landscape(
    technology='Xenium',   # CosMx was converted to the Xenium format
    base_url=base_url,
    creds={
        'accessKeyId': creds.access_key,
        'secretAccessKey': creds.secret_key,
        'sessionToken': creds.token,
    },
    height=700,
)
landscape

### Optional: Landscape + Clustergram (cluster gene-expression signatures)
`df_sig.parquet` is read from the s3fs-mounted workbench path.

In [ ]:
import pandas as pd
df_sig = pd.read_parquet(
    f'/home/jovyan/workbench/CosMx_Celldega/cosmx_dega_files/{SAMPLE}/df_sig.parquet'
)
mat = dega.clust.Matrix(df_sig)
mat.norm(axis='col', by='total')
mat.norm(axis='row', by='zscore')
mat.cluster()
cgm = dega.viz.Clustergram(matrix=mat, width=500, height=500)
dega.viz.landscape_clustergram(landscape, cgm)